
# 03B. Modelos baseline com conjunto reduzido e robusto de features

Este notebook usa o dataset enriquecido, mas reduz deliberadamente o número de variáveis para diminuir overfitting e extrapolação.

## Objetivo

Comparar novamente modelos lineares usando um conjunto compacto e estável de features, preservando:

- split temporal por contrato;
- pesos iguais por contrato;
- classificação apenas antes do vencimento do prazo;
- seleção de modelo/threshold somente na validação;
- teste reservado para avaliação final.

## Estratégia de robustez

1. Seleção manual de features economicamente e operacionalmente interpretáveis.
2. Exclusão de médias/taxas históricas de atraso de fornecedor e prédio.
3. Exclusão de variáveis redundantes.
4. Clipping por quantis aprendido **somente no treino**.
5. `RobustScaler` no lugar de `StandardScaler`.
6. Regularização mais forte para Ridge e Logistic Regression.

O notebook continua comparando tudo com modelos Dummy.


In [1]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42


## 1. Localização do dataset enriquecido

In [2]:

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path.resolve()

    return None


DATASET_PATH = first_existing([
    PROJECT_ROOT / "data/processed/dataset_modelagem_obras_snapshots_enriquecido.csv",
    PROJECT_ROOT / "modelagem_outputs/processed/dataset_modelagem_obras_snapshots_enriquecido.csv",
    PROJECT_ROOT / "dataset_modelagem_obras_snapshots_enriquecido.csv",
    Path("/mnt/data/modelagem_outputs/processed/dataset_modelagem_obras_snapshots_enriquecido.csv"),
    Path("/mnt/data/data/processed/dataset_modelagem_obras_snapshots_enriquecido.csv"),
])

if DATASET_PATH is None:
    raise FileNotFoundError(
        "dataset_modelagem_obras_snapshots_enriquecido.csv não encontrado."
    )


REPORTS_DIR = Path("/mnt/data/03B_modelos_robustos_outputs")
REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Dataset:", DATASET_PATH)
print("Saídas:", REPORTS_DIR.resolve())


Dataset: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots_enriquecido.csv
Saídas: C:\mnt\data\03B_modelos_robustos_outputs


## 2. Carga e validações básicas

In [3]:

df = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
    encoding="utf-8-sig",
)

date_columns = [
    "data_corte",
    "data_os",
    "data_termino_original",
    "data_termino_vigente_corte",
    "data_conclusao_definitiva",
    "data_conclusao_controle",
    "data_termino_vigente_final_controle",
]

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(
            df[column],
            errors="coerce",
        )


required = [
    "id_contrato",
    "data_corte",
    "target_dias_atraso",
    "target_atraso",
    "flag_apto_treino",
    "flag_apto_classificacao",
    "sample_weight_contrato",
    "sample_weight_classificacao",
    "split_modelagem",
    "dias_restantes_prazo_vigente",
    "data_conclusao_definitiva",
    "data_termino_vigente_final_controle",
]

missing = [
    column
    for column in required
    if column not in df.columns
]

if missing:
    raise ValueError(
        f"Colunas obrigatórias ausentes: {missing}"
    )


reg_data = df[
    df["flag_apto_treino"].eq(1)
    & df["split_modelagem"].isin(
        ["train", "validation", "test"]
    )
].copy()

clf_data = df[
    df["flag_apto_classificacao"].eq(1)
    & df["split_modelagem"].isin(
        ["train", "validation", "test"]
    )
].copy()


assert (
    reg_data.groupby("id_contrato")[
        "split_modelagem"
    ].nunique().max()
    == 1
)

assert (
    clf_data.groupby("id_contrato")[
        "split_modelagem"
    ].nunique().max()
    == 1
)

assert (
    reg_data["data_corte"]
    < reg_data["data_conclusao_definitiva"]
).all()

assert (
    clf_data[
        "dias_restantes_prazo_vigente"
    ].ge(0)
).all()

print("Shape original:", df.shape)
print("Contratos:", df["id_contrato"].nunique())
print("Snapshots regressão:", len(reg_data))
print("Snapshots classificação:", len(clf_data))


Shape original: (633, 109)
Contratos: 48
Snapshots regressão: 585
Snapshots classificação: 509



## 3. Conjunto reduzido de features

O conjunto abaixo evita:

- médias/taxas históricas de atraso por fornecedor ou prédio;
- múltiplas janelas 30/60/90 dias da mesma variável;
- pares fortemente redundantes;
- IDs de alta cardinalidade;
- datas finais e controles;
- ano/mês absolutos, que podem induzir tendência temporal espúria.

Mantemos apenas variáveis estáveis de contrato, prazo, aditivos, execução, parcelas e financeiro.


In [ ]:

NUMERIC_FEATURES_REQUESTED = [
    # contrato
    "prazo_original_dias",
    "log_valor_total_contrato",
    "perc_desconto_modelo",
    "dias_assinatura_ate_os",

    # histórico simples sem target
    "fornecedor_contratos_anteriores",
    "predio_contratos_anteriores",

    # estágio atual
    "dias_desde_os",
    "dias_restantes_prazo_vigente",
    "pct_prazo_consumido",

    # aditivos
    "qtd_aditivos_prazo",
    "pct_dias_aditados_sobre_prazo_original",
    "qtd_aditivos_valor",

    # execução física / parcelas
    "pct_parcelas_concluidas_vs_previstas",
    "atraso_medio_parcelas_dias",
    "pct_parcelas_atrasadas",
    "dias_desde_ultima_parcela_concluida",

    # financeiro
    "pct_financeiro_liberado",
    "atraso_medio_liberacao_dias",

    # interações operacionais
    "gap_prazo_execucao",
    "gap_prazo_financeiro",
]

CATEGORICAL_FEATURES_REQUESTED = [
    "classificacao_contrato",
    "id_setor_divisao",
]


numeric_features = [
    c
    for c in NUMERIC_FEATURES_REQUESTED
    if c in df.columns
]

categorical_features = [
    c
    for c in CATEGORICAL_FEATURES_REQUESTED
    if c in df.columns
]

features = (
    numeric_features
    + categorical_features
)

missing_requested = sorted(
    set(
        NUMERIC_FEATURES_REQUESTED
        + CATEGORICAL_FEATURES_REQUESTED
    )
    - set(features)
)

if missing_requested:
    print(
        "Features solicitadas ausentes e ignoradas:",
        missing_requested,
    )

if not features:
    raise ValueError(
        "Nenhuma feature selecionada."
    )


LEAKAGE_BLOCKLIST = {
    "id_contrato",
    "data_corte",
    "data_os" # data bruta pode fazer o modelo aprender efeitos de calendário/época, por exemplo, que obras iniciadas em 2024 se comportam diferente das iniciadas em 2020
    "data_termino_original",
    "data_termino_vigente_corte",
    "status_final_controle",
    "data_conclusao_sistema_controle",
    "data_conclusao_definitiva",
    "data_conclusao_controle",
    "data_termino_vigente_final_controle",
    "diferenca_vigente_original_final_controle",
    "target_dias_atraso",
    "target_atraso",
    "flag_apto_treino",
    "flag_apto_classificacao",
    "sample_weight_contrato",
    "sample_weight_classificacao",
    "split_contrato_temporal",
    "split_modelagem",
}

assert not (
    set(features)
    & LEAKAGE_BLOCKLIST
)


# Remove feature constante no treino.
train_reg = reg_data[
    reg_data["split_modelagem"].eq("train")
]

numeric_features = [
    c
    for c in numeric_features
    if train_reg[c].dropna().nunique() > 1
]

categorical_features = [
    c
    for c in categorical_features
    if train_reg[c].dropna().nunique() > 1
]

features = (
    numeric_features
    + categorical_features
)

print("Features numéricas:", len(numeric_features))
print("Features categóricas:", len(categorical_features))
print("Total:", len(features))
print()
print(features)


Features numéricas: 20
Features categóricas: 2
Total: 22

['prazo_original_dias', 'log_valor_total_contrato', 'perc_desconto_modelo', 'dias_assinatura_ate_os', 'fornecedor_contratos_anteriores', 'predio_contratos_anteriores', 'dias_desde_os', 'dias_restantes_prazo_vigente', 'pct_prazo_consumido', 'qtd_aditivos_prazo', 'pct_dias_aditados_sobre_prazo_original', 'qtd_aditivos_valor', 'pct_parcelas_concluidas_vs_previstas', 'atraso_medio_parcelas_dias', 'pct_parcelas_atrasadas', 'dias_desde_ultima_parcela_concluida', 'pct_financeiro_liberado', 'atraso_medio_liberacao_dias', 'gap_prazo_execucao', 'gap_prazo_financeiro', 'classificacao_contrato', 'id_setor_divisao']


## 4. Diagnóstico de estabilidade treino × validação × teste

In [5]:

stability_rows = []

for feature in numeric_features:
    train_series = (
        reg_data.loc[
            reg_data["split_modelagem"].eq("train"),
            feature,
        ]
        .dropna()
        .astype(float)
    )

    if train_series.empty:
        continue

    q01 = train_series.quantile(0.01)
    q99 = train_series.quantile(0.99)
    median = train_series.median()
    iqr = (
        train_series.quantile(0.75)
        - train_series.quantile(0.25)
    )

    for split in [
        "validation",
        "test",
    ]:
        split_series = (
            reg_data.loc[
                reg_data["split_modelagem"].eq(split),
                feature,
            ]
            .dropna()
            .astype(float)
        )

        if split_series.empty:
            continue

        stability_rows.append({
            "feature": feature,
            "split": split,
            "train_q01": q01,
            "train_q99": q99,
            "split_min": split_series.min(),
            "split_max": split_series.max(),
            "pct_fora_q01_q99_treino": (
                (
                    (split_series < q01)
                    | (split_series > q99)
                ).mean()
            ),
            "dist_mediana_em_iqr": (
                abs(
                    split_series.median()
                    - median
                )
                / iqr
                if iqr > 0
                else np.nan
            ),
        })

stability = pd.DataFrame(
    stability_rows
)

display(
    stability.sort_values(
        [
            "pct_fora_q01_q99_treino",
            "dist_mediana_em_iqr",
        ],
        ascending=False,
    ).head(20)
)


,feature,split,train_q01,train_q99,split_min,split_max,pct_fora_q01_q99_treino,dist_mediana_em_iqr
1,prazo_original_dias,test,60.0000,720.0000,60.0000,960.0000,0.4497,1.2000
27,atraso_medio_parcelas_dias,test,-157.5126,46.0000,-577.6667,28.2000,0.2598,0.0629
3,log_valor_total_contrato,test,11.2177,16.7534,12.8386,16.8013,0.2483,1.6616
5,perc_desconto_modelo,test,0.0000,20.6000,0.0000,25.4600,0.2483,0.0530
15,dias_restantes_prazo_vigente,test,-256.7700,643.6700,-142.0000,960.0000,0.1141,0.6817
9,fornecedor_contratos_anteriores,test,0.0000,3.0000,0.0000,4.0000,0.1007,0.0000
4,perc_desconto_modelo,validation,0.0000,20.6000,0.5000,31.6600,0.0469,0.0816
20,pct_dias_aditados_sobre_prazo_original,validation,0.0000,1.0838,0.0000,1.3917,0.0469,0.0000
21,pct_dias_aditados_sobre_prazo_original,test,0.0000,1.0838,0.0000,2.2500,0.0403,0.0000
13,dias_desde_os,test,0.0000,972.6700,0.0000,"1,092.0000",0.0268,0.4690



## 5. Preprocessamento

`QuantileClipper` aprende limites no treino e aplica clipping nos outros conjuntos.

Depois:

- imputação pela mediana;
- `RobustScaler`, baseado em mediana e intervalo interquartil.

Isso reduz a influência de valores muito extremos sem usar informação de validação ou teste.


In [6]:

class QuantileClipper(
    BaseEstimator,
    TransformerMixin,
):
    def __init__(
        self,
        lower=0.01,
        upper=0.99,
    ):
        self.lower = lower
        self.upper = upper

    def fit(
        self,
        X,
        y=None,
    ):
        arr = np.asarray(
            X,
            dtype=float,
        )

        self.lower_bounds_ = np.nanquantile(
            arr,
            self.lower,
            axis=0,
        )

        self.upper_bounds_ = np.nanquantile(
            arr,
            self.upper,
            axis=0,
        )

        return self

    def transform(
        self,
        X,
    ):
        arr = np.asarray(
            X,
            dtype=float,
        ).copy()

        return np.clip(
            arr,
            self.lower_bounds_,
            self.upper_bounds_,
        )


numeric_pipeline = Pipeline([
    (
        "clipper",
        QuantileClipper(
            lower=0.01,
            upper=0.99,
        ),
    ),
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
        ),
    ),
    (
        "scaler",
        RobustScaler(),
    ),
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent",
        ),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
        ),
    ),
])

preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
)


def split_regression(
    data,
    split,
):
    part = data[
        data["split_modelagem"].eq(split)
    ].copy()

    return (
        part,
        part[features],
        part[
            "target_dias_atraso"
        ].astype(float),
        part[
            "sample_weight_contrato"
        ].astype(float),
    )


def split_classification(
    data,
    split,
):
    part = data[
        data["split_modelagem"].eq(split)
    ].copy()

    return (
        part,
        part[features],
        part[
            "target_atraso"
        ].astype(int),
        part[
            "sample_weight_classificacao"
        ].astype(float),
    )


# Parte A — Regressão

In [7]:

reg_train, Xr_train, yr_train, wr_train = split_regression(
    reg_data,
    "train",
)

reg_val, Xr_val, yr_val, wr_val = split_regression(
    reg_data,
    "validation",
)

reg_test, Xr_test, yr_test, wr_test = split_regression(
    reg_data,
    "test",
)


regression_models = {
    "dummy_mediana": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            DummyRegressor(
                strategy="median",
            ),
        ),
    ]),

    "ridge_alpha_10": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            Ridge(
                alpha=10.0,
            ),
        ),
    ]),

    "ridge_alpha_100": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            Ridge(
                alpha=100.0,
            ),
        ),
    ]),

    "ridge_alpha_1000": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            Ridge(
                alpha=1000.0,
            ),
        ),
    ]),
}


def regression_metrics(
    y_true,
    y_pred,
    sample_weight,
):
    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.maximum(
        0,
        np.asarray(
            y_pred,
            dtype=float,
        ),
    )

    error = (
        y_pred
        - y_true
    )

    return {
        "mae_ponderado": mean_absolute_error(
            y_true,
            y_pred,
            sample_weight=sample_weight,
        ),
        "medae": median_absolute_error(
            y_true,
            y_pred,
        ),
        "rmse_ponderado": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
                sample_weight=sample_weight,
            )
        ),
        "r2_ponderado": r2_score(
            y_true,
            y_pred,
            sample_weight=sample_weight,
        ),
        "pct_erro_abs_ate_30d": np.average(
            np.abs(error) <= 30,
            weights=sample_weight,
        ),
        "erro_assinado_medio": np.average(
            error,
            weights=sample_weight,
        ),
    }


reg_models_fitted = {}
reg_results = []

for name, model in regression_models.items():
    model.fit(
        Xr_train,
        yr_train,
        model__sample_weight=wr_train,
    )

    reg_models_fitted[name] = model

    for split_name, X, y, w in [
        (
            "validation",
            Xr_val,
            yr_val,
            wr_val,
        ),
        (
            "test",
            Xr_test,
            yr_test,
            wr_test,
        ),
    ]:
        pred = np.maximum(
            0,
            model.predict(X),
        )

        reg_results.append({
            "modelo": name,
            "split": split_name,
            **regression_metrics(
                y,
                pred,
                w,
            ),
        })


reg_results = pd.DataFrame(
    reg_results
)

display(
    reg_results.sort_values(
        [
            "split",
            "mae_ponderado",
        ]
    )
)

BEST_REGRESSION_MODEL = (
    reg_results[
        reg_results["split"].eq(
            "validation"
        )
    ]
    .sort_values(
        [
            "mae_ponderado",
            "rmse_ponderado",
        ]
    )
    .iloc[0]["modelo"]
)

print(
    "Melhor regressão:",
    BEST_REGRESSION_MODEL,
)


,modelo,split,mae_ponderado,medae,rmse_ponderado,r2_ponderado,pct_erro_abs_ate_30d,erro_assinado_medio
1,dummy_mediana,test,21.8750,0.0000,54.3519,-0.1933,0.8750,-21.8750
7,ridge_alpha_1000,test,53.9123,33.0247,73.9499,-1.2090,0.4833,25.5506
5,ridge_alpha_100,test,62.1650,43.2208,88.3203,-2.1509,0.4917,34.3499
3,ridge_alpha_10,test,69.5484,59.3781,97.5560,-2.8444,0.5000,40.3584
0,dummy_mediana,validation,12.7143,14.0000,28.8370,-0.2413,0.8571,-12.7143
6,ridge_alpha_1000,validation,54.4456,64.3377,85.2288,-9.8430,0.5714,35.0914
4,ridge_alpha_100,validation,61.7559,62.4754,100.5431,-14.0898,0.5714,41.5839
2,ridge_alpha_10,validation,63.9746,74.7144,103.7790,-15.0767,0.5714,43.2178


Melhor regressão: dummy_mediana


In [16]:
# ============================================================
# AMOSTRA DE PREVISÕES — NOTEBOOK 03
# Uma linha por contrato, snapshot mais próximo de 90 dias
# ============================================================

HORIZONTE_AMOSTRA = 90

# ------------------------------------------------------------
# CLASSIFICAÇÃO
# ------------------------------------------------------------

amostra_03 = clf_test[
    [
        "id_contrato",
        "data_corte",
        "dias_restantes_prazo_vigente",
        "target_atraso",
        "target_dias_atraso",
    ]
].copy()

amostra_03["probabilidade_atraso"] = prob_test
amostra_03["previsto_atraso"] = pred_test


# ------------------------------------------------------------
# REGRESSÃO
# ------------------------------------------------------------

reg_pred = np.maximum(
    0,
    best_reg.predict(Xr_test)
)

reg_aux = reg_test[
    [
        "id_contrato",
        "data_corte",
    ]
].copy()

# IMPORTANTE:
# não adicionamos target_dias_atraso novamente,
# porque essa coluna já existe em amostra_03.
reg_aux["prev_dias_atraso"] = reg_pred


# ------------------------------------------------------------
# JUNTA CLASSIFICAÇÃO + PREVISÃO DE DIAS
# ------------------------------------------------------------

amostra_03 = amostra_03.merge(
    reg_aux,
    on=[
        "id_contrato",
        "data_corte",
    ],
    how="left",
    validate="one_to_one",
)


# ------------------------------------------------------------
# SNAPSHOT MAIS PRÓXIMO DE 90 DIAS
# ------------------------------------------------------------

amostra_03["dist_horizonte"] = (
    amostra_03["dias_restantes_prazo_vigente"]
    - HORIZONTE_AMOSTRA
).abs()

amostra_03 = (
    amostra_03
    .sort_values(
        [
            "id_contrato",
            "dist_horizonte",
            "data_corte",
        ]
    )
    .groupby(
        "id_contrato",
        as_index=False,
    )
    .first()
)


# ------------------------------------------------------------
# FORMATAÇÃO FINAL
# ------------------------------------------------------------

amostra_03 = amostra_03[
    [
        "id_contrato",
        "data_corte",
        "dias_restantes_prazo_vigente",

        "target_atraso",
        "probabilidade_atraso",
        "previsto_atraso",

        "target_dias_atraso",
        "prev_dias_atraso",
    ]
].copy()

amostra_03 = amostra_03.rename(
    columns={
        "target_atraso": "real_atraso",
        "target_dias_atraso": "real_dias_atraso",
    }
)

amostra_03["probabilidade_atraso"] = (
    amostra_03["probabilidade_atraso"]
    .round(3)
)

amostra_03["prev_dias_atraso"] = (
    amostra_03["prev_dias_atraso"]
    .round(1)
)


print("=" * 90)
print(
    f"AMOSTRA DE PREVISÕES — TESTE "
    f"≈ {HORIZONTE_AMOSTRA} DIAS ANTES DO PRAZO"
)
print("=" * 90)

print("Threshold:", round(BEST_THRESHOLD, 2))
print("Regressor:", BEST_REGRESSION_MODEL)

display(amostra_03)

AMOSTRA DE PREVISÕES — TESTE ≈ 90 DIAS ANTES DO PRAZO
Threshold: 0.05
Regressor: dummy_mediana


,id_contrato,data_corte,dias_restantes_prazo_vigente,real_atraso,probabilidade_atraso,previsto_atraso,real_dias_atraso,prev_dias_atraso
0,173,2024-07-31,80,0,0.4850,1,0.0000,0.0000
1,201,2024-11-30,102,0,0.4850,1,0.0000,0.0000
2,254,2025-05-31,83,0,0.4850,1,0.0000,0.0000
3,283,2024-11-30,97,0,0.4850,1,0.0000,0.0000
4,287,2025-03-31,84,0,0.4850,1,0.0000,0.0000
5,317,2024-11-30,84,1,0.4850,1,23.0000,0.0000
6,342,2025-07-18,60,0,0.4850,1,0.0000,0.0000
7,377,2025-04-29,90,1,0.4850,1,152.0000,0.0000


## 6. Avaliação da regressão por contrato e horizonte

In [8]:

best_reg = reg_models_fitted[
    BEST_REGRESSION_MODEL
]

reg_test_eval = reg_test.copy()

reg_test_eval[
    "pred_dias_atraso"
] = np.maximum(
    0,
    best_reg.predict(
        Xr_test
    ),
)


def select_nearest_horizon(
    data,
    horizon,
):
    subset = data[
        data[
            "dias_restantes_prazo_vigente"
        ].ge(0)
    ].copy()

    subset[
        "dist_horizonte"
    ] = (
        subset[
            "dias_restantes_prazo_vigente"
        ]
        - horizon
    ).abs()

    return (
        subset
        .sort_values(
            [
                "id_contrato",
                "dist_horizonte",
                "data_corte",
            ]
        )
        .groupby(
            "id_contrato",
            as_index=False,
        )
        .first()
    )


reg_horizon_rows = []

for horizon in [
    180,
    90,
    60,
    30,
]:
    h = select_nearest_horizon(
        reg_test_eval,
        horizon,
    )

    reg_horizon_rows.append({
        "horizonte_dias": horizon,
        "contratos": h[
            "id_contrato"
        ].nunique(),
        "mae_contrato": (
            mean_absolute_error(
                h[
                    "target_dias_atraso"
                ],
                h[
                    "pred_dias_atraso"
                ],
            )
        ),
        "rmse_contrato": np.sqrt(
            mean_squared_error(
                h[
                    "target_dias_atraso"
                ],
                h[
                    "pred_dias_atraso"
                ],
            )
        ),
    })


reg_horizon_results = pd.DataFrame(
    reg_horizon_rows
)

display(reg_horizon_results)


,horizonte_dias,contratos,mae_contrato,rmse_contrato
0,180,8,21.8750,54.3519
1,90,8,21.8750,54.3519
2,60,8,21.8750,54.3519
3,30,8,21.8750,54.3519


# Parte B — Classificação

In [9]:

clf_train, Xc_train, yc_train, wc_train = split_classification(
    clf_data,
    "train",
)

clf_val, Xc_val, yc_val, wc_val = split_classification(
    clf_data,
    "validation",
)

clf_test, Xc_test, yc_test, wc_test = split_classification(
    clf_data,
    "test",
)


for name, part, y in [
    (
        "train",
        clf_train,
        yc_train,
    ),
    (
        "validation",
        clf_val,
        yc_val,
    ),
    (
        "test",
        clf_test,
        yc_test,
    ),
]:
    contract_level = (
        part
        .sort_values("data_corte")
        .drop_duplicates(
            "id_contrato"
        )
    )

    print(
        name,
        "| snapshots:",
        len(part),
        "| contratos:",
        part[
            "id_contrato"
        ].nunique(),
        "| taxa snapshot:",
        f"{y.mean():.1%}",
        "| taxa contrato:",
        f"{contract_level['target_atraso'].mean():.1%}",
    )


classification_models = {
    "dummy_prior": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            DummyClassifier(
                strategy="prior",
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "logistic_C_0_01": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            LogisticRegression(
                C=0.01,
                max_iter=5000,
                solver="liblinear",
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "logistic_C_0_1": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            LogisticRegression(
                C=0.1,
                max_iter=5000,
                solver="liblinear",
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "logistic_balanced_C_0_01": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            LogisticRegression(
                C=0.01,
                max_iter=5000,
                solver="liblinear",
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "logistic_balanced_C_0_1": Pipeline([
        (
            "preprocessing",
            clone(preprocessing),
        ),
        (
            "model",
            LogisticRegression(
                C=0.1,
                max_iter=5000,
                solver="liblinear",
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]),
}


def safe_auc(
    y_true,
    prob,
    sample_weight=None,
):
    if pd.Series(
        y_true
    ).nunique() < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        prob,
        sample_weight=sample_weight,
    )


def safe_ap(
    y_true,
    prob,
    sample_weight=None,
):
    if pd.Series(
        y_true
    ).nunique() < 2:
        return np.nan

    return average_precision_score(
        y_true,
        prob,
        sample_weight=sample_weight,
    )


def classification_metrics(
    y_true,
    prob,
    threshold,
    sample_weight,
):
    prob = np.asarray(
        prob,
        dtype=float,
    )

    pred = (
        prob >= threshold
    ).astype(int)

    return {
        "threshold": threshold,
        "accuracy_ponderada": accuracy_score(
            y_true,
            pred,
            sample_weight=sample_weight,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            pred,
            sample_weight=sample_weight,
        ),
        "precision": precision_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
        "brier": brier_score_loss(
            y_true,
            prob,
            sample_weight=sample_weight,
        ),
        "roc_auc": safe_auc(
            y_true,
            prob,
            sample_weight,
        ),
        "pr_auc": safe_ap(
            y_true,
            prob,
            sample_weight,
        ),
    }


clf_models_fitted = {}
clf_results_05 = []

for name, model in classification_models.items():
    model.fit(
        Xc_train,
        yc_train,
        model__sample_weight=wc_train,
    )

    clf_models_fitted[
        name
    ] = model

    for split_name, X, y, w in [
        (
            "validation",
            Xc_val,
            yc_val,
            wc_val,
        ),
        (
            "test",
            Xc_test,
            yc_test,
            wc_test,
        ),
    ]:
        prob = (
            model.predict_proba(
                X
            )[:, 1]
        )

        clf_results_05.append({
            "modelo": name,
            "split": split_name,
            **classification_metrics(
                y,
                prob,
                0.50,
                w,
            ),
        })


clf_results_05 = pd.DataFrame(
    clf_results_05
)

display(
    clf_results_05.sort_values(
        [
            "split",
            "roc_auc",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


train | snapshots: 311 | contratos: 33 | taxa snapshot: 44.1% | taxa contrato: 48.5%
validation | snapshots: 61 | contratos: 7 | taxa snapshot: 50.8% | taxa contrato: 28.6%
test | snapshots: 137 | contratos: 8 | taxa snapshot: 11.7% | taxa contrato: 25.0%


,modelo,split,threshold,accuracy_ponderada,balanced_accuracy,precision,recall,f1,brier,roc_auc,pr_auc
7,logistic_balanced_C_0_01,test,0.5000,0.5867,0.6272,0.3422,0.7083,0.4615,0.2363,0.5870,0.2982
3,logistic_C_0_01,test,0.5000,0.5963,0.6337,0.3487,0.7083,0.4673,0.2343,0.5791,0.2931
9,logistic_balanced_C_0_1,test,0.5000,0.5551,0.6201,0.3290,0.7500,0.4574,0.2975,0.5384,0.2717
5,logistic_C_0_1,test,0.5000,0.5593,0.6229,0.3314,0.7500,0.4597,0.2952,0.5354,0.2700
1,dummy_prior,test,0.5000,0.7500,0.5000,0.0000,0.0000,0.0000,0.2427,0.5000,0.2500
0,dummy_prior,validation,0.5000,0.7143,0.5000,0.0000,0.0000,0.0000,0.2437,0.5000,0.2857
2,logistic_C_0_01,validation,0.5000,0.3149,0.2205,0.0000,0.0000,0.0000,0.2858,0.2683,0.2032
6,logistic_balanced_C_0_01,validation,0.5000,0.2911,0.2038,0.0000,0.0000,0.0000,0.2912,0.2594,0.2013
8,logistic_balanced_C_0_1,validation,0.5000,0.2662,0.2136,0.0519,0.0909,0.0661,0.4071,0.2490,0.2015
4,logistic_C_0_1,validation,0.5000,0.2597,0.2023,0.0395,0.0682,0.0500,0.4034,0.2424,0.1998


## 7. Seleção do classificador e threshold

In [10]:

validation_clf = (
    clf_results_05[
        clf_results_05[
            "split"
        ].eq(
            "validation"
        )
    ]
    .copy()
)


if validation_clf[
    "roc_auc"
].notna().any():
    validation_clf = (
        validation_clf
        .sort_values(
            [
                "roc_auc",
                "pr_auc",
                "balanced_accuracy",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
    )
else:
    validation_clf = (
        validation_clf
        .sort_values(
            [
                "balanced_accuracy",
                "f1",
            ],
            ascending=[
                False,
                False,
            ],
        )
    )


BEST_CLASSIFICATION_MODEL = (
    validation_clf
    .iloc[0][
        "modelo"
    ]
)

best_clf = (
    clf_models_fitted[
        BEST_CLASSIFICATION_MODEL
    ]
)

prob_val = (
    best_clf.predict_proba(
        Xc_val
    )[:, 1]
)


threshold_rows = []

for threshold in np.arange(
    0.05,
    0.951,
    0.01,
):
    threshold_rows.append(
        classification_metrics(
            yc_val,
            prob_val,
            threshold,
            wc_val,
        )
    )


threshold_results = pd.DataFrame(
    threshold_rows
)


BEST_THRESHOLD = float(
    threshold_results
    .sort_values(
        [
            "balanced_accuracy",
            "f1",
            "precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .iloc[0][
        "threshold"
    ]
)


prob_test = (
    best_clf.predict_proba(
        Xc_test
    )[:, 1]
)

pred_test = (
    prob_test
    >= BEST_THRESHOLD
).astype(int)


final_classification = pd.DataFrame([
    {
        "split": "validation",
        **classification_metrics(
            yc_val,
            prob_val,
            BEST_THRESHOLD,
            wc_val,
        ),
    },
    {
        "split": "test",
        **classification_metrics(
            yc_test,
            prob_test,
            BEST_THRESHOLD,
            wc_test,
        ),
    },
])


print(
    "Melhor classificador:",
    BEST_CLASSIFICATION_MODEL,
)

print(
    "Threshold:",
    round(
        BEST_THRESHOLD,
        2,
    ),
)

display(
    final_classification
)

print(
    "Matriz de confusão — teste:"
)

print(
    confusion_matrix(
        yc_test,
        pred_test,
    )
)


Melhor classificador: dummy_prior
Threshold: 0.05


,split,threshold,accuracy_ponderada,balanced_accuracy,precision,recall,f1,brier,roc_auc,pr_auc
0,validation,0.0500,0.2857,0.5000,0.2857,1.0000,0.4444,0.2437,0.5000,0.2857
1,test,0.0500,0.2500,0.5000,0.2500,1.0000,0.4000,0.2427,0.5000,0.2500


Matriz de confusão — teste:
[[  0 121]
 [  0  16]]


## 8. Classificação por contrato em horizontes operacionais

In [11]:

clf_test_eval = (
    clf_test.copy()
)

clf_test_eval[
    "prob_atraso"
] = prob_test

clf_test_eval[
    "pred_atraso"
] = pred_test


def horizon_metrics(
    data,
    horizon,
):
    subset = data.copy()

    subset[
        "dist_horizonte"
    ] = (
        subset[
            "dias_restantes_prazo_vigente"
        ]
        - horizon
    ).abs()

    subset = (
        subset
        .sort_values(
            [
                "id_contrato",
                "dist_horizonte",
                "data_corte",
            ]
        )
        .groupby(
            "id_contrato",
            as_index=False,
        )
        .first()
    )

    if subset[
        "target_atraso"
    ].nunique() > 1:
        bac = balanced_accuracy_score(
            subset[
                "target_atraso"
            ],
            subset[
                "pred_atraso"
            ],
        )

        auc = roc_auc_score(
            subset[
                "target_atraso"
            ],
            subset[
                "prob_atraso"
            ],
        )

        ap = average_precision_score(
            subset[
                "target_atraso"
            ],
            subset[
                "prob_atraso"
            ],
        )

    else:
        bac = np.nan
        auc = np.nan
        ap = np.nan

    return {
        "horizonte_dias": horizon,
        "contratos": subset[
            "id_contrato"
        ].nunique(),
        "taxa_atraso": subset[
            "target_atraso"
        ].mean(),
        "balanced_accuracy": bac,
        "roc_auc": auc,
        "pr_auc": ap,
    }, subset


rows = []
detail_90 = None

for horizon in [
    180,
    90,
    60,
    30,
]:
    result, detail = horizon_metrics(
        clf_test_eval,
        horizon,
    )

    rows.append(
        result
    )

    if horizon == 90:
        detail_90 = detail


clf_horizon_results = pd.DataFrame(
    rows
)

display(
    clf_horizon_results
)

print(
    "Detalhe ~90 dias:"
)

display(
    detail_90[
        [
            "id_contrato",
            "data_corte",
            "dias_restantes_prazo_vigente",
            "target_atraso",
            "prob_atraso",
            "pred_atraso",
        ]
    ]
)


,horizonte_dias,contratos,taxa_atraso,balanced_accuracy,roc_auc,pr_auc
0,180,8,0.2500,0.5000,0.5000,0.2500
1,90,8,0.2500,0.5000,0.5000,0.2500
2,60,8,0.2500,0.5000,0.5000,0.2500
3,30,8,0.2500,0.5000,0.5000,0.2500


Detalhe ~90 dias:


,id_contrato,data_corte,dias_restantes_prazo_vigente,target_atraso,prob_atraso,pred_atraso
0,173,2024-07-31,80,0,0.4848,1
1,201,2024-11-30,102,0,0.4848,1
2,254,2025-05-31,83,0,0.4848,1
3,283,2024-11-30,97,0,0.4848,1
4,287,2025-03-31,84,0,0.4848,1
5,317,2024-11-30,84,1,0.4848,1
6,342,2025-07-18,60,0,0.4848,1
7,377,2025-04-29,90,1,0.4848,1


## 9. Comparação de overfitting treino × validação

In [12]:

overfit_rows = []

for name, model in clf_models_fitted.items():
    if name == "dummy_prior":
        continue

    prob_train = (
        model.predict_proba(
            Xc_train
        )[:, 1]
    )

    prob_validation = (
        model.predict_proba(
            Xc_val
        )[:, 1]
    )

    overfit_rows.append({
        "modelo": name,
        "roc_auc_train": safe_auc(
            yc_train,
            prob_train,
            wc_train,
        ),
        "roc_auc_validation": safe_auc(
            yc_val,
            prob_validation,
            wc_val,
        ),
        "gap_auc_train_validation": (
            safe_auc(
                yc_train,
                prob_train,
                wc_train,
            )
            - safe_auc(
                yc_val,
                prob_validation,
                wc_val,
            )
        ),
    })


overfit_results = pd.DataFrame(
    overfit_rows
)

display(
    overfit_results.sort_values(
        "gap_auc_train_validation"
    )
)


,modelo,roc_auc_train,roc_auc_validation,gap_auc_train_validation
0,logistic_C_0_01,0.7600,0.2683,0.4918
2,logistic_balanced_C_0_01,0.7599,0.2594,0.5004
3,logistic_balanced_C_0_1,0.8400,0.2490,0.5910
1,logistic_C_0_1,0.8398,0.2424,0.5974


## 10. Exportação

In [13]:

REG_PATH = (
    REPORTS_DIR
    / "03B_regressao_features_reduzidas.csv"
)

REG_HORIZON_PATH = (
    REPORTS_DIR
    / "03B_regressao_horizontes.csv"
)

CLF_PATH = (
    REPORTS_DIR
    / "03B_classificacao_threshold_050.csv"
)

CLF_FINAL_PATH = (
    REPORTS_DIR
    / "03B_classificacao_final.csv"
)

CLF_HORIZON_PATH = (
    REPORTS_DIR
    / "03B_classificacao_horizontes.csv"
)

THRESHOLD_PATH = (
    REPORTS_DIR
    / "03B_thresholds_validacao.csv"
)

STABILITY_PATH = (
    REPORTS_DIR
    / "03B_estabilidade_features.csv"
)

OVERFIT_PATH = (
    REPORTS_DIR
    / "03B_overfitting_classificacao.csv"
)

FEATURES_PATH = (
    REPORTS_DIR
    / "03B_features_utilizadas.csv"
)


reg_results.to_csv(
    REG_PATH,
    index=False,
    encoding="utf-8-sig",
)

reg_horizon_results.to_csv(
    REG_HORIZON_PATH,
    index=False,
    encoding="utf-8-sig",
)

clf_results_05.to_csv(
    CLF_PATH,
    index=False,
    encoding="utf-8-sig",
)

final_classification.to_csv(
    CLF_FINAL_PATH,
    index=False,
    encoding="utf-8-sig",
)

clf_horizon_results.to_csv(
    CLF_HORIZON_PATH,
    index=False,
    encoding="utf-8-sig",
)

threshold_results.to_csv(
    THRESHOLD_PATH,
    index=False,
    encoding="utf-8-sig",
)

stability.to_csv(
    STABILITY_PATH,
    index=False,
    encoding="utf-8-sig",
)

overfit_results.to_csv(
    OVERFIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame({
    "feature": features,
    "tipo": [
        (
            "categorica"
            if f in categorical_features
            else "numerica"
        )
        for f in features
    ],
}).to_csv(
    FEATURES_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("=" * 72)
print("NOTEBOOK 03B — FEATURES REDUZIDAS E ROBUSTAS")
print("=" * 72)

print(
    "Features utilizadas:",
    len(features),
)

print(
    "Melhor regressão:",
    BEST_REGRESSION_MODEL,
)

print(
    "Melhor classificação:",
    BEST_CLASSIFICATION_MODEL,
)

print(
    "Threshold:",
    round(
        BEST_THRESHOLD,
        2,
    ),
)

print(
    "Saídas:",
    REPORTS_DIR.resolve(),
)

print(
    "Sanity checks concluídos."
)


NOTEBOOK 03B — FEATURES REDUZIDAS E ROBUSTAS
Features utilizadas: 22
Melhor regressão: dummy_mediana
Melhor classificação: dummy_prior
Threshold: 0.05
Saídas: C:\mnt\data\03B_modelos_robustos_outputs
Sanity checks concluídos.
